In [1]:
# ==============================================================================
# TRUE MULTI-CLASS INTRUSION DETECTION (ANN BASE MODEL)
#
# Project: Base Model for Meta-Learning Ensemble
# Author: B12 - BPPIMT - CSE 21-25
# Date: 12-06-25
#
# Description:
#   This script trains a true multi-class Artificial Neural Network (ANN) on
#   the CIC-IDS2018 dataset. It correctly handles the multiple attack classes,
#   balances the data, and exports all artifacts required for use as a base
#   model in a meta-learning (stacking) ensemble.
# ==============================================================================

# ==============================================================================
# 1. SETUP AND IMPORTS
# ==============================================================================
import numpy as np
import pandas as pd
import os
import gc
import json
import joblib
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')

# ==============================================================================
# 2. GLOBAL CONFIGURATION
# ==============================================================================
DATASET_PATH = "/kaggle/input/cse-cic-ids2018/CSE-CIC-IDS2018"
OUTPUT_PATH = "./ann_multiclass_model/"
os.makedirs(OUTPUT_PATH, exist_ok=True)
RANDOM_STATE = 42
TEST_SIZE = 0.2
MIN_SAMPLES_FOR_TRAINING = 100 # Increased for more stable multi-class training

EPOCHS = 50
BATCH_SIZE = 2048 # Larger batch size for faster training
PATIENCE = 10

# ==============================================================================
# 3. DATA LOADING AND PREPARATION
# ==============================================================================
print("\n[INFO] Step 3: Loading and consolidating data...")
try:
    csv_files = [f for f in os.listdir(DATASET_PATH) if f.endswith('.csv')]
    df_list = [pd.read_csv(os.path.join(DATASET_PATH, f), low_memory=False) for f in csv_files]
    df = pd.concat(df_list, ignore_index=True)
    del df_list; gc.collect()
except Exception as e:
    print(f"Error loading data: {e}"); exit()

# --- Data Cleaning ---
df.columns = df.columns.str.strip()
if 'Timestamp' in df.columns:
    df = df.drop('Timestamp', axis=1)
for col in df.columns:
    if df[col].dtype == object and col != 'Label':
        df[col] = pd.to_numeric(df[col], errors='coerce')
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna(0, inplace=True)
print("Data cleaning complete.")

# ==============================================================================
# 4. FEATURE AND LABEL ENGINEERING (FOR MULTI-CLASS)
# ==============================================================================
print("\n[INFO] Step 4: Engineering features and labels for multi-class classification...")
y = df['Label']
X = df.drop(columns=['Label'])
del df; gc.collect()

# --- Filter Rare Classes ---
original_label_counts = y.value_counts()
eligible_classes = original_label_counts[original_label_counts >= MIN_SAMPLES_FOR_TRAINING].index.tolist()
print(f"Filtering to classes with at least {MIN_SAMPLES_FOR_TRAINING} samples.")
X = X[y.isin(eligible_classes)]
y = y[y.isin(eligible_classes)]
print(f"Eligible classes for training: {list(y.unique())}")


# --- EXPORT FEATURE NAMES (CRITICAL FOR META-MODEL) ---
feature_names = X.columns.tolist()
with open(os.path.join(OUTPUT_PATH, 'feature_names.json'), 'w') as f:
    json.dump(feature_names, f)
print("✅ Feature names mapping saved successfully.")

# --- Encode Labels and Scale Features ---
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
num_classes = len(label_encoder.classes_)
joblib.dump(label_encoder, os.path.join(OUTPUT_PATH, 'label_encoder.pkl'))
print(f"Label encoder saved. Found {num_classes} eligible classes.")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
joblib.dump(scaler, os.path.join(OUTPUT_PATH, 'scaler.pkl'))
print("Scaler saved.")
del X; gc.collect()

# ==============================================================================
# 5. DATA SPLITTING
# ==============================================================================
print("\n[INFO] Step 5: Splitting data into training and testing sets...")
X_train_raw, X_test, y_train_raw, y_test = train_test_split(
    X_scaled, y_encoded, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_encoded
)
del X_scaled, y_encoded; gc.collect()

# ==============================================================================
# 6. MULTI-CLASS BALANCING WITH PIPELINE
# ==============================================================================
print("\n[INFO] Step 6: Handling multi-class imbalance...")
y_train_raw_counts = pd.Series(y_train_raw).value_counts()
print(f"Class distribution before resampling:\n{y_train_raw_counts.sort_index()}")

# Define a strategy to bring all classes to a reasonable, consistent count
TARGET_COUNT = 40000
# Strategy for SMOTE: any class with fewer than TARGET_COUNT samples
over_strategy = {cls: TARGET_COUNT for cls, count in y_train_raw_counts.items() if count < TARGET_COUNT and count > 1}
# Strategy for RandomUnderSampler: any class with more than TARGET_COUNT samples
under_strategy = {cls: TARGET_COUNT for cls, count in y_train_raw_counts.items() if count > TARGET_COUNT}

# Check if strategies are empty
if not over_strategy and not under_strategy:
    print("No resampling needed based on defined strategies.")
    X_train, y_train = X_train_raw, y_train_raw
else:
    print(f"Oversampling classes to {TARGET_COUNT}. Undersampling classes to {TARGET_COUNT}.")
    over = SMOTE(sampling_strategy=over_strategy, random_state=RANDOM_STATE, n_jobs=-1)
    under = RandomUnderSampler(sampling_strategy=under_strategy, random_state=RANDOM_STATE)
    pipeline = Pipeline(steps=[('o', over), ('u', under)], verbose=True)
    try:
        X_train, y_train = pipeline.fit_resample(X_train_raw, y_train_raw)
        print("Resampling pipeline applied successfully.")
    except (ValueError, MemoryError) as e:
        print(f"WARNING: Resampling failed: {e}. Using original unbalanced data.");
        X_train, y_train = X_train_raw, y_train_raw

del X_train_raw; gc.collect()
print(f"\nFinal training set shape: {X_train.shape}")
print(f"Final training distribution:\n{pd.Series(y_train).value_counts().sort_index()}")

# ==============================================================================
# 7. ANN MODEL TRAINING
# ==============================================================================
print("\n[INFO] Step 7: Building and training the Multi-Class ANN model...")
input_dim = X_train.shape[1]
model = Sequential([
    Dense(128, activation='relu', input_shape=(input_dim,)),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax') # Correct output layer for multi-class
])
# Correct loss function for integer-based multi-class labels
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

early_stopping = EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True, verbose=1)
model_checkpoint = ModelCheckpoint(os.path.join(OUTPUT_PATH, 'best_threat_detector_model.h5'), save_best_only=True, monitor='val_loss', verbose=1)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping, model_checkpoint],
    verbose=1
)
print("Model training complete.")

# ==============================================================================
# 8. MODEL EVALUATION AND ARTIFACT EXPORT
# ==============================================================================
print("\n[INFO] Step 8: Evaluating performance and exporting all artifacts...")
best_model = load_model(os.path.join(OUTPUT_PATH, 'best_threat_detector_model.h5'))

y_pred_probs = best_model.predict(X_test, batch_size=BATCH_SIZE)
y_pred_encoded = np.argmax(y_pred_probs, axis=1)
all_target_names = [str(label) for label in label_encoder.classes_]

# --- Save Reports and Visualizations ---
report = classification_report(y_test, y_pred_encoded, target_names=all_target_names, digits=4, zero_division=0, output_dict=True)
print("\n--- Detailed Classification Report ---")
print(pd.DataFrame(report).transpose())
with open(os.path.join(OUTPUT_PATH, 'classification_report.json'), 'w') as f:
    json.dump(report, f, indent=4)

conf_matrix = confusion_matrix(y_test, y_pred_encoded, labels=np.arange(num_classes))
plt.figure(figsize=(14, 12))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=all_target_names, yticklabels=all_target_names)
plt.title('Multi-Class Confusion Matrix Heatmap'); plt.xlabel('Predicted Label'); plt.ylabel('True Label')
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_PATH, 'confusion_matrix_heatmap.png')); plt.close()

plt.figure(figsize=(12, 5)); plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy'); plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training & Validation Accuracy'); plt.legend(); plt.grid(True)
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss'); plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training & Validation Loss'); plt.legend(); plt.grid(True)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_PATH, 'training_history.png')); plt.close()

# ==============================================================================
# ✨ NEW CODE BLOCK: GENERATE BINARY (BENIGN VS. ATTACK) CONFUSION MATRIX
# ==============================================================================
print("\n[INFO] Generating aggregated 'Benign vs. Attack' confusion matrix...")
try:
    # Find the integer index that corresponds to the 'Benign' class
    benign_label_index = list(label_encoder.classes_).index('Benign')
    
    # Create binary labels: 0 for Benign, 1 for Attack
    # If a label is the benign_index, map to 0, otherwise map to 1 (Attack)
    y_test_binary = np.where(y_test == benign_label_index, 0, 1)
    y_pred_binary = np.where(y_pred_encoded == benign_label_index, 0, 1)
    
    # Define the labels for the 2x2 matrix
    binary_target_names = ['Benign', 'Attack']
    
    # Calculate the 2x2 confusion matrix
    binary_conf_matrix = confusion_matrix(y_test_binary, y_pred_binary)
    
    # Plot the heatmap
    plt.figure(figsize=(8, 6))
    sns.heatmap(binary_conf_matrix, annot=True, fmt='d', cmap='Greens',
                xticklabels=binary_target_names, yticklabels=binary_target_names)
    plt.title('Binary Confusion Matrix (Benign vs. Attack)')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_PATH, 'binary_confusion_matrix_heatmap.png'))
    plt.close()
    
    print("\n✅ Binary (Benign vs. Attack) confusion matrix saved successfully.")

except ValueError:
    print("\n[WARNING] 'Benign' class not found. Skipping binary confusion matrix generation.")
except Exception as e:
    print(f"\n[ERROR] Could not generate binary confusion matrix: {e}")

# ==============================================================================

print("\n[SUCCESS] All artifacts for the Multi-Class ANN base model have been saved.")


[INFO] Step 3: Loading and consolidating data...
Data cleaning complete.

[INFO] Step 4: Engineering features and labels for multi-class classification...
Filtering to classes with at least 100 samples.
Eligible classes for training: ['Benign', 'Brute Force -Web', 'DoS attacks-Slowloris', 'DDoS attacks-LOIC-HTTP', 'Brute Force -XSS', 'Bot', 'DDOS attack-HOIC', 'Infilteration', 'DDOS attack-LOIC-UDP', 'SSH-Bruteforce', 'DoS attacks-GoldenEye', 'FTP-BruteForce', 'DoS attacks-Hulk', 'DoS attacks-SlowHTTPTest']
✅ Feature names mapping saved successfully.
Label encoder saved. Found 14 eligible classes.
Scaler saved.

[INFO] Step 5: Splitting data into training and testing sets...

[INFO] Step 6: Handling multi-class imbalance...
Class distribution before resampling:
0     5501530
1      228953
2         489
3         184
4      548809
5        1384
6      460953
7       33206
8      369530
9      111912
10       8792
11     154688
12     129547
13     150071
Name: count, dtype: int64
Overs